# Run I: train on the realistic-style data, then score everything (Colab)
Runtime → Change runtime type → **T4 GPU**.

Upload to the Files sidebar and wait for each upload to finish: `fallacy_code.zip` (code, tiny) and `external_eval.zip` (CoCoLoFa and MAFALDA test files, **private, local use only**). Needs the Colab secret `HF_TOKEN` (read token) with *Notebook access* on. Data: private dataset `sreejithb/fallacy-dataset`, version `v5`.

Run I uses run B's settings (rank 32, 5 epochs, lr 2e-4, batch 8, no validation holdout). Only the training data changes: the earlier sets plus the new realistic-style `train_real.jsonl`. Expect roughly 1.5-2 hours in total on a T4 (training about 1-1.5 hours, scoring about 25 minutes). Keep the tab open.

In [ ]:
!nvidia-smi -L
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v5
!wc -l data/generated/*.jsonl external/*.jsonl

## Train run I (expect about 6,000 training examples)

In [ ]:
!python train.py --out out/lora_I --epochs 5 --rank 32 --val-per-label 0 2>&1 | tee train_I.log

## Quick evaluation (strict accuracy, false-alarm rate)

In [ ]:
!python evaluate.py --adapter out/lora_I --out lora_I_preds.jsonl
!python evaluate.py --adapter out/lora_I --eval-file data/generated/eval_none.jsonl --out none_I_preds.jsonl
!python evaluate.py --adapter out/lora_I --eval-file data/generated/eval_real.jsonl --out real_I_preds.jsonl

## Score every name on every set (for calibration)

In [ ]:
!python score_candidates.py --adapter out/lora_I --run I \
  --set eval=data/generated/eval.jsonl \
  --set eval_none=data/generated/eval_none.jsonl \
  --set eval_real=data/generated/eval_real.jsonl \
  --set calib_fallacy=data/generated/calib_fallacy.jsonl \
  --set calib_none=data/generated/calib_none.jsonl \
  --set cocolofa_calib=external/cocolofa_calib.jsonl \
  --set cocolofa_test=external/cocolofa_test.jsonl \
  --set mafalda_test=external/mafalda_test.jsonl \
  --set cocolofa_tradition_calib=external/cocolofa_tradition_calib.jsonl \
  --set cocolofa_tradition_test=external/cocolofa_tradition_test.jsonl

In [ ]:
!zip -qr result_I.zip out/lora_I data/scores_I_*.json data/lora_I_preds.jsonl data/none_I_preds.jsonl data/real_I_preds.jsonl train_I.log -x '*/README.md'
from google.colab import files
files.download('/content/work/result_I.zip')